In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q soundfile librosa
!pip install -q deep-filter
!pip install -q pesq pystoi
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载DeepFilterNet代码
import os
if not os.path.exists('DeepFilterNet-main'):
    print('正在克隆DeepFilterNet仓库...')
    !git clone https://github.com/Rikorose/DeepFilterNet.git DeepFilterNet-main
    print('克隆完成！')

# 下载预训练模型
model_dir = 'DeepFilterNet-main/models'
if not os.path.exists(os.path.join(model_dir, 'DeepFilterNet3')):
    print('正在下载预训练模型...')
    !cd DeepFilterNet-main/models && unzip -q DeepFilterNet3.zip 2>/dev/null || echo '需要手动上传模型权重'

# 生成测试音频
print('环境准备就绪！')


# Module 5 Session 1: Speech Enhancement Basics + DeepFilterNet PaperThis notebook covers:1. Speech enhancement problem definition and signal model2. Traditional methods (Spectral Subtraction, Wiener Filter)3. Evaluation metrics (PESQ, STOI, SI-SDR)4. DeepFilterNet paper structured reading guide---

## Section 1: Speech Enhancement Problem Definition**Signal Model:**- x(t) = s(t) + n(t)- x(t): noisy speech (observed signal)- s(t): clean speech (target signal)- n(t): noise (interference to remove)**Goal of speech enhancement**: estimate s(t) from x(t), minimizing |s_hat(t) - s(t)|.In the CI context:- x(t) = noisy speech picked up by CI microphone- s(t) = ideal clean speech CI user should receive- Speech enhancement is front-end processing, ACE strategy is back-end encoding

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Demo: generate noisy speechsr = 16000duration = 3.0t = np.linspace(0, duration, int(sr * duration), endpoint=False)# Generate clean speech (multi-harmonic)f0 = 150clean = np.zeros_like(t)for h in range(1, 10):    clean += (0.5 / h) * np.sin(2 * np.pi * f0 * h * t)clean = clean / np.max(np.abs(clean)) * 0.8# Generate noisenp.random.seed(42)noise = np.random.randn(len(t)) * 0.3def mix_at_snr(clean, noise, snr_db):    clean_power = np.mean(clean ** 2)    noise_power = np.mean(noise ** 2)    snr_linear = 10 ** (snr_db / 10)    noise_scaled = noise * np.sqrt(clean_power / (noise_power * snr_linear + 1e-8))    return clean + noise_scalednoisy_0db = mix_at_snr(clean, noise, 0)noisy_5db = mix_at_snr(clean, noise, 5)noisy_10db = mix_at_snr(clean, noise, 10)print("Signal shape:", clean.shape)print("Clean energy:", np.mean(clean**2))

## Section 2: Traditional Speech Enhancement Methods### 2.1 Spectral SubtractionSimplest speech enhancement method:- |S_hat(f)|^2 = |X(f)|^2 - |N_hat(f)|^2- Drawback: produces "musical noise" (random spectral spots)### 2.2 Wiener FilterOptimal linear filter:- H(f) = P_ss(f) / (P_ss(f) + P_nn(f))- P_ss(f): clean speech power spectrum- P_nn(f): noise power spectrum- H(f): frequency-dependent gain functionLimitations: assumes signal and noise are uncorrelated, requires power spectrum estimation

In [ ]:
# Demo: Spectral Subtraction and Wiener Filterfrom scipy.fft import fft, ifftN = len(clean)clean_fft = fft(clean)noise_fft = fft(noise)noisy_fft = fft(noisy_0db)noise_power = np.abs(noise_fft) ** 2clean_power = np.abs(clean_fft) ** 2# Spectral Subtractionestimated_power = np.abs(noisy_fft) ** 2 - noise_powerestimated_power = np.maximum(estimated_power, 0)estimated_fft = np.sqrt(estimated_power) * np.exp(1j * np.angle(noisy_fft))enhanced_spectral = np.real(ifft(estimated_fft))# Wiener Filterwiener_gain = clean_power / (clean_power + noise_power + 1e-8)enhanced_wiener = np.real(ifft(wiener_gain * noisy_fft))fig, axes = plt.subplots(4, 1, figsize=(12, 8))axes[0].plot(t[:1600], clean[:1600]); axes[0].set_title("Clean Speech")axes[1].plot(t[:1600], noisy_0db[:1600]); axes[1].set_title("Noisy (0 dB SNR)")axes[2].plot(t[:1600], enhanced_spectral[:1600]); axes[2].set_title("Spectral Subtraction")axes[3].plot(t[:1600], enhanced_wiener[:1600]); axes[3].set_title("Wiener Filter")for ax in axes:    ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")plt.tight_layout()plt.show()

## Section 3: Speech Enhancement Evaluation Metrics| Metric | Full Name | Measures | Range | CI Relevance ||--------|-----------|----------|-------|-------------|| PESQ | Perceptual Evaluation of Speech Quality | Perceptual quality | -0.5 ~ 4.5 | CI user quality perception || STOI | Short-Time Objective Intelligibility | Speech intelligibility | 0 ~ 1 | How much CI users can understand || SI-SDR | Scale-Invariant SDR | Signal distortion | -inf ~ +inf dB | Overall enhancement quality |> **Key insight**: CI users care about **intelligibility** (STOI), not just **quality** (PESQ). Enhancement must make speech understandable, not just pleasant-sounding.

In [ ]:
# Compute evaluation metricsdef compute_si_sdr(estimated, reference):    reference = reference - np.mean(reference)    estimated = estimated - np.mean(estimated)    alpha = np.dot(estimated, reference) / (np.dot(reference, reference) + 1e-8)    s_target = alpha * reference    e_noise = estimated - s_target    si_sdr = 10 * np.log10(np.dot(s_target, s_target) / (np.dot(e_noise, e_noise) + 1e-8))    return si_sdrmethods = {"Spectral Subtraction": enhanced_spectral, "Wiener Filter": enhanced_wiener}print("Evaluation Metrics Comparison (0 dB SNR input):")print("-" * 50)for name, enhanced in methods.items():    si_sdr = compute_si_sdr(enhanced, clean)    print("  %s: SI-SDR = %.1f dB" % (name, si_sdr))si_sdr_input = compute_si_sdr(noisy_0db, clean)print("\nInput (0dB SNR): SI-SDR = %.1f dB" % si_sdr_input)print("\nFor full evaluation: pip install pesq pystoi")

## Section 4: Metrics Deep Dive### PESQ- Based on human perception model, simulates subjective MOS scoring- Range: -0.5 to 4.5 (higher is better)- Measures **quality**, not necessarily intelligibility### STOI- Based on short-time spectral correlation- Range: 0 to 1 (higher is better)- Measures **intelligibility**, highly correlated with ASR accuracy- **Critical for CI**: CI users care most about understanding speech### SI-SDR- Signal-level distortion metric- Unit: dB (higher is better)- Differentiable, can be used as training loss- Does not consider perception> **Think about it**: Why might improving SI-SDR not always improve STOI?

## Section 5: DeepFilterNet Paper Reading Guide### 5.1 Structured Reading Card**Pass 1: Title and Abstract (5 min)**- [ ] What problem does DeepFilterNet solve?- [ ] What is "Deep Filtering"? How does it differ from traditional filtering?- [ ] Why use ERB (Equivalent Rectangular Bandwidth) domain instead of Mel?**Pass 2: Introduction (10 min)**- [ ] What are the limitations of traditional speech enhancement?- [ ] Why is deep learning better suited for speech enhancement?- [ ] What are DeepFilterNet's core innovations?**Pass 3: Method (20 min)**- [ ] Why use two stages (Enhancement + Deep Filtering)?- [ ] What is the difference between ERB domain and STFT domain?- [ ] What is the mathematical formula for Deep Filtering?- [ ] What loss functions are used?**Pass 4: Experiments (10 min)**- [ ] What training data is used? How are noisy-clean pairs constructed?- [ ] What datasets and metrics are used for evaluation?- [ ] How does it compare to other methods?

### 5.2 DeepFilterNet Architecture Overview**Stage 1: Enhancement**- Input: noisy spectrogram (STFT)- ERB feature extraction -> Encoder (2D Conv) -> GroupedGRU -> ErbDecoder- Output: ERB-domain mask -> applied to spectrogram -> pre-enhanced spectrogram**Stage 2: Deep Filtering**- Pre-enhanced spectrogram- DF branch in Encoder -> DfDecoder (GroupedGRU) -> DF coefficients- DF coefficients perform temporal filtering on low frequencies: Y[f] = sum_k(C[k] * X[f-k])- alpha controls DF mixing ratio> Key insight: ERB (Equivalent Rectangular Bandwidth) simulates human frequency resolution, intrinsically linked to CI electrode mapping.

### 5.3 Key Innovations vs Known Knowledge| DeepFilterNet Component | Known Knowledge | Innovation ||------------------------|----------------|------------|| ERB feature extraction | Module 3 Mel spectrogram | ERB scale models human hearing resolution || Enhancement stage | Module 2 spectrogram classification | Outputs mask instead of class labels || Deep Filtering | Traditional FIR filter | Network learns filter coefficients || GroupedGRU | Module 3 GRU | Splits GRU into groups, reduces parameters || Two-stage design | -- | Coarse enhancement then fine refinement || alpha parameter | -- | Controls DF stage participation, adaptive switching |

### 5.4 ERB vs Mel Scale- Mel scale: based on pure tone perception, common in ASR and sound classification- ERB scale: based on noise band perception, closer to actual human frequency resolution- CI electrode mapping: 22 electrodes ~ 22 ERB bands- DeepFilterNet: 32 ERB bands (finer but still CI-compatible)> ERB scale relates to CI more directly than Mel -- CI electrode frequency allocation follows a nonlinear mapping similar to ERB.

## Section 6: Discussion Questions1. **Methodology**: Traditional spectral subtraction and Wiener filter use fixed rules (statistical models) for enhancement, while deep learning uses data-driven approaches. What are the pros and cons of each paradigm?2. **Metric Selection**: If you were evaluating a speech enhancement system for CI users, would you prioritize PESQ or STOI? Why?3. **Deep Filtering**: Why does DF only operate on low frequencies? How does this relate to speech signal characteristics?4. **Relationship with ACE**: If DeepFilterNet is placed before ACE (enhance then encode), how would this affect ACE channel selection?5. **Real-time Performance**: DeepFilterNet is designed for real-time operation (~5ms algorithmic latency). Why is real-time performance critical for CI/hearing aid scenarios?---## SummaryThis session covered:1. Speech enhancement signal model and problem definition2. Traditional methods (spectral subtraction, Wiener filter) and their limitations3. Three core evaluation metrics (PESQ, STOI, SI-SDR)4. Structured reading of DeepFilterNet paper5. Connection between ERB domain and CI electrode mapping**Homework**:- Complete all questions in the paper reading guide- Draw the complete DeepFilterNet data flow diagram- Prepare for next session: read `DeepFilterNet/df/` code files